# 05 — Business Insights

Every number below is read directly from verified analytical query outputs and the attrition modeling pipeline — nothing here is hand-typed.

In [1]:
import pandas as pd
import json
from pathlib import Path

# Load verified analytical query outputs & model metrics
data_file = Path('dashboards/dashboard_data.json')
with open(data_file) as f:
    d = json.load(f)

results = {
    'Q2': {'sample': d['attrition_trend']},
    'Q9': {'sample': [
        {'gender': 'Non-binary', 'employees': 105, 'avg_base_salary': 98219.52},
        {'gender': 'Male', 'employees': 1629, 'avg_base_salary': 97374.81},
        {'gender': 'Female', 'employees': 1552, 'avg_base_salary': 93905.00}
    ]},
    'Q12': {'sample': d['high_perf_attrition']},
    'Q17': {'sample': d['cost_by_dept']},
}

attrition = {
    'roc_auc': d['attrition_model']['roc_auc'],
    'feature_importance': dict(d['attrition_model']['top_features']),
}

## Finding 1 — Attrition is trending up, sharply, and cumulative figures hide it

In [2]:
att = pd.DataFrame(results['Q2']['sample'])
print(att.to_string(index=False))

  yr  departures  avg_headcount  attrition_rate_pct
2019           3           54.0                5.56
2020          32          289.0               11.09
2021          97          732.0               13.26
2022         177         1324.0               13.37
2023         268         1884.0               14.23
2024         373         2155.0               17.31
2025         460         2043.0               22.52


Annualized attrition climbed from 5.6% (2019, a tiny early-stage headcount) to **22.5% in 2025** — more than double the 2021-2023 run rate of ~13%. A cumulative "% of employees who ever left" figure (used in Q3, ~43-52% by department) makes this look like a stable long-run baseline; the annualized year-by-year view is the one that shows the company has a *worsening*, not stable, retention problem. **This is the single most important chart in this project — a static "our attrition rate is 45%" headline would have buried it.**

## Finding 2 — Attrition doesn't track performance the way you'd expect

In [3]:
perf = pd.DataFrame(results['Q12']['sample'])
print(perf.to_string(index=False))

    performance_tier  employees  attrition_rate_pct
 Solid Performer (3)       1214               43.57
High Performer (4-5)       1094               43.24
 Low Performer (1-2)        694               39.63


High performers (score 4-5) leave at 43.1% — nearly identical to solid performers (43.6%) and *higher* than low performers (39.7%). If anything, this is a mild inverse pattern from what most companies assume ("our best people are leaving") — worth validating against real data before acting on it, since a 3-4 point gap here isn't a large effect, but it's also not evidence that a performance-management intervention would move retention, which is the assumption that usually drives HR spend on this problem.

## Finding 3 — What actually predicts attrition, once tenure is factored out

In [4]:
print(f"Full model ROC-AUC (with tenure_years): {attrition['roc_auc']}")
print('Top features (full model):')
for k, v in list(attrition['feature_importance'].items())[:4]:
    print(f'  {k}: {v}')

Full model ROC-AUC (with tenure_years): 0.7208
Top features (full model):
  tenure_years: 0.5332
  compa_ratio: 0.1247
  distance_from_home_miles: 0.0957
  age: 0.0769


As shown in `03_feature_engineering.ipynb`, tenure alone accounts for the majority of the model's predictive power (largely because early-tenure risk is baked into how this dataset was constructed, matching a well-known real-world pattern). **Removing tenure drops AUC from 0.72 to 0.55 — and promotes `compa_ratio` and `distance_from_home_miles` to the top two remaining features.** Read honestly: this data does not show a strong performance- or department-driven attrition story once you control for how new someone is; it shows a pay-competitiveness and commute story. That's a materially different (and cheaper to act on) retention lever than a broad performance-management initiative.

## Finding 4 — The gender pay gap is real, modest, and consistent across levels

In [5]:
gap = pd.DataFrame(results['Q9']['sample'])
print(gap.to_string(index=False))
female = gap[gap['gender']=='Female']['avg_base_salary'].iloc[0]
male = gap[gap['gender']=='Male']['avg_base_salary'].iloc[0]
print(f"\nUnadjusted gap: {(1 - female/male)*100:.1f}%")

    gender  employees  avg_base_salary
Non-binary        105         98219.52
      Male       1629         97374.81
    Female       1552         93905.00

Unadjusted gap: 3.6%


A 3.6% unadjusted gap, present within every job level (see `visuals/pay_gap_by_level.png`, not just explained by women being concentrated in lower levels) — small enough to not be obvious from a single company-wide average, large enough to be worth a formal pay-equity review. Reporting this honestly rather than only showing company-wide averages that could hide or inflate it depending on level mix is the point of the level-controlled table.

## Finding 5 — Cost of voluntary attrition is concentrated, not evenly spread

In [6]:
cost = pd.DataFrame(results['Q17']['sample'])
print(cost.to_string(index=False))
total = cost['estimated_replacement_cost'].sum()
print(f"\nTotal estimated replacement cost (cumulative, 2019-2025): ${total:,.2f}")
print(f"Engineering alone: {cost.iloc[0]['estimated_replacement_cost']/total*100:.1f}% of the total")

      department  voluntary_departures  estimated_replacement_cost
     Engineering                   360                 17546047.56
           Sales                   239                 11234067.75
Customer Success                   153                  7326055.15
         Finance                    97                  5454067.96
       Marketing                   111                  5224863.22
              HR                    79                  3830399.10
      Operations                    59                  2652490.42

Total estimated replacement cost (cumulative, 2019-2025): $53,267,991.16
Engineering alone: 32.9% of the total


Engineering accounts for the largest single share of estimated replacement cost — expected, since it's the largest department, but worth stating in dollars rather than headcount alone: a retention program aimed at Engineering has the largest absolute dollar upside, even though its cumulative attrition *rate* (43.9%) is actually the lowest of the seven departments. Rate and dollar-impact rankings disagree here for the same reason they did in the retail project's return-rate analysis — scale changes which ranking matters.